# TFM — IDRecovery Grid Search en Kaggle

**Cómo usarlo**
1. Sube la carpeta `tfm/` como *Dataset* y añádelo al notebook (panel derecho → **Add Input**).
2. Ejecuta las celdas en orden. No hace falta tocar rutas.

**Salida:** los CSV se escriben *incrementalmente* (una fila por combinación) en
`/kaggle/working/resultados/`, que es lo que Kaggle guarda como **Output**. Así,
si vuelves más tarde, los resultados parciales ya están ahí y no se pierden.

In [ ]:
# === Celda 1: preparar entorno ===
import sys, os, shutil, glob

# Localiza la carpeta 'tfm' a cualquier nivel dentro de /kaggle/input
hits = [d for d in glob.glob("/kaggle/input/**/tfm", recursive=True) if os.path.isdir(d)]
if not hits:
    print("No encuentro 'tfm'. Esto es lo que hay montado en /kaggle/input:")
    for p in sorted(glob.glob("/kaggle/input/*")):
        print(" ", p, "->", os.listdir(p)[:6])
    raise SystemExit("Añade el dataset (Add Input) o revisa la estructura de arriba.")

src = hits[0]
print("Código encontrado en:", src)

# /kaggle/input es de SOLO LECTURA -> copiamos a /kaggle/working para poder escribir.
shutil.copytree(src, "/kaggle/working/tfm", dirs_exist_ok=True)
os.chdir("/kaggle/working/tfm")
sys.path.insert(0, "/kaggle/working/tfm")

# Carpeta de salida dedicada DENTRO del Output de Kaggle.
OUT_DIR = "/kaggle/working/resultados"
os.makedirs(OUT_DIR, exist_ok=True)

print("CWD:", os.getcwd())
print("Contenido tfm:", os.listdir("."))
print("Salida CSV en:", OUT_DIR)

In [ ]:
# === Celda 2: dependencias ===
# pysmile (.so) y su licencia vienen DENTRO del dataset, no se instalan por pip.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "EDAspy", "psutil", "pybnesian", "pgmpy"], check=True)

import pysmile_license, pysmile
print("pysmile OK, version", getattr(pysmile, "__version__", "?"))

In [ ]:
# === Celda 3: lanzar el grid search (los 4 optimizadores en paralelo) ===
# -u            -> salida sin buffer, progreso en vivo (Repetición 1/5, 2/5...).
# --base_folder -> apunta al Output: cada combinación se escribe nada más terminar.
import subprocess, sys
subprocess.run([sys.executable, "-u", "grid_search.py",
    "--optimizer_type", "all",
    "--xdsl_path",   "example/nhlv1/network-nhlv1.xdsl",
    "--rules_csv",   "example/nhlv1/reglas_generadas.csv",
    "--base_folder", "/kaggle/working/resultados",
], check=True)

In [ ]:
# === Celda 4: inspeccionar resultados ===
import pandas as pd, glob
for f in sorted(glob.glob("/kaggle/working/resultados/grid_search_results_*.csv")):
    print("\n=====", f, "=====")
    display(pd.read_csv(f))

## Si el run no termina (límite de tiempo de Kaggle)

- Los CSV de `/kaggle/working/resultados/` se rellenan **combinación a combinación**.
  En sesión **interactiva** puedes descargarlos cuando quieras desde el panel
  *Data / Output* (refresca) — no hace falta esperar al final.
- `grid_search.py` tiene **checkpointing**: al relanzarlo relee los CSV y **salta**
  las combinaciones ya hechas. Para reanudar en una sesión nueva, vuelve a colocar
  los CSV parciales en `/kaggle/working/resultados/` antes de ejecutar la Celda 3.